In [0]:
from pyspark.sql.functions import (
    count,
    sum,
    avg,
    round,
    col,
    when
)

FACT_TABLE = "workspace.default.fact_trip"

df_fact = spark.table(FACT_TABLE)

print("Fact rows:", df_fact.count())

In [0]:
df_shared_gold = (
    df_fact
    .groupBy("provider_key")
    .agg(
        count("*").alias("total_trips"),

        sum(
            when(col("shared_request") == True, 1)
            .otherwise(0)
        ).alias("shared_requests"),

        sum(
            when(col("shared_match") == True, 1)
            .otherwise(0)
        ).alias("shared_matches")
    )
)

In [0]:
df_shared_gold = (
    df_shared_gold
    .withColumn(
        "shared_request_rate",
        round(
            when(
                col("total_trips") > 0,
                col("shared_requests") /
                col("total_trips") * 100
            ),
            2
        )
    )
    .withColumn(
        "shared_match_rate",
        round(
            when(
                col("total_trips") > 0,
                col("shared_matches") /
                col("total_trips") * 100
            ),
            2
        )
    )
)

In [0]:
df_provider = spark.table(
    "workspace.default.dim_provider"
)

df_shared_gold = (
    df_shared_gold
    .join(
        df_provider.select(
            "provider_key",
            "provider_name"
        ),
        on="provider_key",
        how="left"
    )
)

In [0]:
display(
    df_shared_gold.orderBy("provider_key")
)

In [0]:
print(
    "Gold shared trips:",
    df_shared_gold
    .select(sum("total_trips"))
    .collect()[0][0]
)

print(
    "Fact trips:",
    df_fact.count()
)

In [0]:
print(
    "Shared requests:",
    df_shared_gold
    .select(sum("shared_requests"))
    .collect()[0][0]
)

print(
    "Shared matches:",
    df_shared_gold
    .select(sum("shared_matches"))
    .collect()[0][0]
)

In [0]:
GOLD_SHARED_TABLE = "workspace.default.gold_shared_ride_metrics"

(
    df_shared_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_SHARED_TABLE)
)

In [0]:
df_shared_final = spark.table(
    "workspace.default.gold_shared_ride_metrics"
)

print("Persisted rows:", df_shared_final.count())

print(
    "Persisted trips:",
    df_shared_final
    .select(sum("total_trips"))
    .collect()[0][0]
)

display(
    df_shared_final.orderBy("provider_key")
)

In [0]:
import sys

SRC_PATH = "/Workspace/Users/gbpatil2002@gmail.com/nyc-hvfhv-data-platform/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from transformation.gold import build_daily_provider_gold

In [0]:
import importlib
import transformation.gold

importlib.reload(transformation.gold)

from transformation.gold import build_shared_ride_gold

In [0]:
df_shared_gold = build_shared_ride_gold(
    spark,
    "workspace.default.fact_trip",
    "workspace.default.dim_provider",
    "workspace.default.gold_shared_ride_metrics"
)

In [0]:
display(df_shared_gold)

In [0]:
from pyspark.sql import functions as F
print(
    "Shared requests:",
    df_shared_gold
    .agg(F.sum("shared_requests"))
    .first()[0]
)

print(
    "Shared matches:",
    df_shared_gold
    .agg(F.sum("shared_matches"))
    .first()[0]
)